In [97]:
import torch
import torch.nn as nn
from torchvision.models import resnet101
import torch.optim as optim
import torch.nn.functional as F
from torchvision.models import resnet18

class CustomResNet18(nn.Module):
    def __init__(self):
        super(CustomResNet18, self).__init__()
        model = resnet18(pretrained=True)
        self.features = nn.Sequential(*list(model.children())[:-1])  # Remove the last layer (fc)
        self.fc = nn.Linear(model.fc.in_features, 256)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)  # Flatten all dimensions except batch
        x = self.fc(x)
        return x

class TripletNet(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        # return self

    def forward(self, anchor, positive, negative):
        anchor[anchor == -1] = 0
        positive[positive == -1] = 0
        negative[negative == -1] = 0
        a = F.normalize(self.backbone(anchor), p=2, dim=1)
        p = F.normalize(self.backbone(positive), p=2, dim=1)
        n = F.normalize(self.backbone(negative), p=2, dim=1)
        # print(self.backbone(anchor))
        # print("Anchor min/max:", a.min(), anchor.max())
        # print("Anchor has NaNs?", torch.isnan(a).any())
        return a, p, n

In [151]:
model = CustomResNet18()
new_model = TripletNet(model)
new_model.load_state_dict(torch.load("triplet_finetuned_r18_2_30_loss_000_min.pth", map_location=torch.device('cuda' if torch.cuda.is_available() else 'cpu')))
new_model.eval()  # Set to evaluation mode

/scratch/4617945.1.csgpu/ipykernel_2767001/862718954.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  new_model.load_state_dict(torch.load("triplet_finetuned_r18_2_30_los

TripletNet(
  (backbone): CustomResNet18(
    (features): Sequential(
      (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (4): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(6

In [152]:
fin_model = new_model.backbone

In [153]:
from torchvision import transforms

transform = transforms.Compose([
            transforms.ToTensor(),  # Converts to [0, 1] float tensor
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # To [-1, 1]
        ])


In [154]:
import os
import cv2
input_path_unmasked = '/projectnb/cs585bp/students/dlgirija/gold_ivc/IVC_Project/output_unmask_test'
embeddings =[]
reference_names = []

for image in os.listdir(input_path_unmasked):
    # images
    # faces = model.get(img)
    # if len(faces) == 0:
    #     print(f"No face found in {file}")
    #     continue
    img = cv2.imread(os.path.join(input_path_unmasked, image))
    img = cv2.resize(img, (112, 112))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = transform(img)
    emb = fin_model(img.unsqueeze(0))
    embeddings.append(emb.detach().numpy()[0])
    reference_names.append(image)


In [155]:
len(embeddings[0])

256

In [156]:
import numpy as np

allEmbeds = np.stack(embeddings)

In [157]:
from sklearn.metrics.pairwise import cosine_similarity

input_dir = '/projectnb/cs585bp/students/dlgirija/gold_ivc/IVC_Project/output_mask_test'
for face in os.listdir(input_dir):
    print()
    print()
    print(face)
    if face == '.ipynb_checkpoints':
        continue
    img = cv2.imread(os.path.join(input_dir, face))
    img = cv2.resize(img, (112, 112))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = transform(img)
    query_faces = fin_model(img.unsqueeze(0)).detach().numpy()[0]
    def mse_similarity(query_embedding, allEmbeds):
        diffs = allEmbeds - query_embedding  # shape: (n, d)
        mse_values = np.mean(diffs ** 2, axis=1)  # shape: (n,)
    
        # Compute norms
        # query_norm = np.linalg.norm(query_embedding) + 1e-8  # shape: ()
        # all_norms = np.linalg.norm(allEmbeds, axis=1) + 1e-8  # shape: (n,)
    
        # # Normalize MSE by product of norms
        # normalization_factors = query_norm * all_norms  # shape: (n,)
        # normalized_mse = mse_values / normalization_factors
    
        # return normalized_mse
        return mse_values
    if len(query_faces) == 0:
        print("No face found in query image.")
    else:
        query_embedding = query_faces
    
    
        similarities = mse_similarity(query_embedding, allEmbeds)
    
        # similarities = cosine_similarity([query_embedding], allEmbeds)[0]
    
        # Step 5: Show results
        best_idx = np.argmin(similarities)
        best_match_name = reference_names[best_idx]
        best_score = similarities[best_idx]
    
        print(f"Best match: {best_match_name} (Score: {1/best_score:.4f})")
    
        # Step 6: Show top-K matches
        top_k = 5
        top_indices = similarities.argsort()[:top_k]
        print("\nTop matches:")
        for i in top_indices:
            print(f"{reference_names[i]} - Closeness: {similarities[i]:.4f}")




Ifreen_Mask_2.png
Best match: Angelina-Jolie.jpeg (Score: 227.6795)

Top matches:
Angelina-Jolie.jpeg - Closeness: 0.0044
Katie_Cassidy_2.jpg - Closeness: 0.0048
Girija_1.jpeg - Closeness: 0.0049
Jen_1.jpg - Closeness: 0.0055
Heer_2.jpeg - Closeness: 0.0055


Girija.jpg
Best match: Kristen_1.jpg (Score: 340.6081)

Top matches:
Kristen_1.jpg - Closeness: 0.0029
Jen_1.jpg - Closeness: 0.0047
Gagan.png - Closeness: 0.0068
Jen_2.jpg - Closeness: 0.0073
Angelina-Jolie.jpeg - Closeness: 0.0074


Cruise_Mask.jpg
Best match: Ananya.jpg (Score: 811.5286)

Top matches:
Ananya.jpg - Closeness: 0.0012
Gagan.png - Closeness: 0.0016
Jen_2.jpg - Closeness: 0.0019
Ifreen_1.jpg - Closeness: 0.0021
Kristen_1.jpg - Closeness: 0.0022


Bradd_Mask_.jpg
Best match: Ananya.jpg (Score: 786.2787)

Top matches:
Ananya.jpg - Closeness: 0.0013
Ifreen_1.jpg - Closeness: 0.0015
Jennie.jpg - Closeness: 0.0027
Jen_2.jpg - Closeness: 0.0041
Gagan.png - Closeness: 0.0048


.ipynb_checkpoints


jolie-post.jpg
Best mat